# Weekly S2S anomaly skill (weeks 1-8)

## Purpose

Subseasonal skill of the five initialization configurations following the ESP-Lab S2S protocol (`ESP-Lab/jupyter/s2s_skill`, `esp_lab/diagnostics/s2s_core.py`): weekly lead blocks, anomalies from separate model and observed climatologies, regional lead-time skill curves in the layout of ESP-Lab `2a_regional_acc_skill_ts` and `3a_sst_skill_ts` (ACC and nRMSE by lead week, Dec, May and all starts), ACC with significance, and the skill difference between initialization methods.

## Inputs

As `01_compute_and_plot_lead_skill.ipynb`: daily member fields of every configuration and start date, ERA5/GPCP/NOAA OLR references and their climatology over `CLIM_YEARS`, and the monthly output of the free-running E3SMv3 AMIP runs `AMIP_CASES` (`AMIP_ROOT/<case>/post/{atm,lnd}/180x360_aave/`) for the model climatology.

## Method

`util/fcst_init_skill.py`:

1. Weeks 1-8 = lead days 1-7, ..., 50-56 (7-day means; ensemble mean for the model).
2. Anomalies on the verification days (`util/fcst_init_data.py`). Both climatologies are built the same way for every variable: calendar-month means over `CLIM_YEARS`, interpolated linearly between mid-month values to the verification days. Observed: reference minus the monthly means of the daily reference (ERA5 `1x1/daily`, daily means of the 6-hourly archive; GPCP V3.3 and NOAA OLR daily), so the climatology comes from the same product as the verifying days. Forecast, both treatments of `BIASES`: `"none"` subtracts the observed climatology, so initialization drift and the model mean bias count as error (the full-field forecast: the error starts small and grows to the model bias); `"amip"` subtracts the monthly climatology of the AMIP runs over the same years, removing the model mean bias (the S2S convention of separate model and observed climatologies; with 4 starts per season the hindcasts' own lead-dependent climatology is unavailable). Neither removes the drift: under `"amip"` a forecast still near the observed state in weeks 1-2 is penalized by the opposite of the model bias. Their difference (`s2s_bias_impact_*`) shows the effect of the model mean bias on skill: positive once the forecast has drifted to the model climate, negative in the first weeks; where it crosses zero indicates the drift time scale of each configuration. Gridpoint ACC within a season hardly depends on the choice (the two climatologies differ by a nearly start-independent field). The pressure-level fields are verified at 00Z (the hindcasts keep only 00Z states of them), against all-hours climatologies on both sides (the AMIP runs have no 00Z T/U/Z output), so each 00Z anomaly carries its own system's 00Z departure from the daily mean (ERA5: RMS about 6 m Z500, 0.5 K T850, 0.5 m s-1 U850/U200; a shared pattern that slightly raises pattern ACC, not gridpoint ACC within a season). `BIAS = "loo"` (mean ensemble-mean error over the other starts) is kept for comparison: starts 5 days apart verify on overlapping days, so it leaks their observed anomalies into the forecast anomaly.
3. Pattern ACC (centred, cos-lat weighted) per region, start and week, averaged over starts; MSSS against the climatology forecast.
4. Gridpoint ACC over the start dates of the available seasons pooled (8 samples with both seasons: one-sided t-test critical value 0.62 at 95%; 4 samples with one: 0.90; stippled), and its difference from `BASELINE` over the seasons both configurations have.
5. Regional lead-time skill (ESP-Lab `2a_regional_acc_skill_ts`): per week, the gridpoint ACC and nRMSE over the starts (as in `02b`), reduced over each region: the cos-lat weighted mean of the gridpoint ACC, and the regional nRMSE sqrt(Σw RMSE² / Σw RMSE_clim²) (nRMSE = RMSE / RMSE of the climatology forecast, below 1 beats climatology). ESP-Lab averages the gridpoint nRMSE; with 4 starts a gridpoint whose observed anomalies happen to be small has a huge nRMSE, so the ratio of the area-summed squares is used. Seasons: Dec starts, May starts, and all 8 pooled (each anomaly from its own season's climatology).
6. Regional-index skill (ESP-Lab `3a_sst_skill_ts`): the cos-lat weighted regional mean of the ensemble-mean and observed anomalies per start and week (the regional index), then its ACC over the starts (filled markers above the one-sided 95% critical value: 0.90 for 4 starts, 0.62 for 8; open below) and nRMSE = RMSE / RMS of the observed index. Plotted for `INDEX_REGIONS` (regions whose mean anomaly varies between starts, e.g. CONUS, Niño3.4; a global-mean anomaly hardly does, so any mean offset dominates its nRMSE). The index series figure shows the index by start date at fixed lead windows (`SERIES_WINDOWS`, e.g. weeks 1-2 and 3-4; the 3a DJF series with start dates in place of years), with each configuration's ACC and nRMSE over the starts of each season.

Caveat: ERA5 is the verifying analysis and the source of the HYB_BASE/DA_LND atmospheric initial state, which favours them in week 1.

## Outputs

`data/hind_lead_eval/s2s_weekly/`: `clim_<SEASON>_<VAR>.nc` (reference), `clim_model_<SEASON>_<VAR>.nc` (AMIP), `anom_<BIAS>_<CONFIG>_<SEASON>_<VAR>.nc` (weekly anomalies), `s2s_weekly_scores.csv` (pattern ACC, MSSS, RMSE per bias treatment and week), `s2s_regional_lead_skill.csv` (2a curves), `s2s_index_skill.csv` and `s2s_index_series.csv` (3a skill and series); figures in `figure/hind_lead_eval/s2s_weekly/`, per bias treatment: regional lead-time skill (`s2s_regional_skill_<BIAS>_<VAR>_<REGION>.png`, 2a), regional-index skill and series (`s2s_index_skill_<BIAS>_<VAR>_<REGION>.png`, `s2s_index_series_<BIAS>_<VAR>_<REGION>.png`, 3a), weekly pattern ACC and MSSS (`s2s_weekly_<BIAS>_<REGION>_<METRIC>.png`), gridpoint ACC maps (2 x 4 weeks) per configuration, ΔACC maps vs `BASELINE` (maps pooled over the setup's seasons), the pattern-ACC scorecard (`s2s_scorecard_<BIAS>_<REGION>.png`), and `s2s_bias_impact_<REGION>_<METRIC>.png` (`"amip"` minus `"none"`). The curves and scorecards show the December and May starts side by side. Only the selection of section 7 is plotted: a named setup of `PLOT_SETUPS` (`PLOT_SETUP`: bias treatments, seasons, figure types), `PLOT_VARIABLES` and `PLOT_REGIONS`.

The lead-time skill maps of two-week means (weeks 1-2, 3-4, 5-6, 7-8; ESP-Lab 1a/1b style) are drawn from these anomalies by `02b_compute_and_plot_s2s_leadtime_skill_maps.ipynb`.

## 1. Libraries

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and run options. Edit this cell for normal changes; it holds plain values only, so `papermill -p` overrides work.

In [ ]:
INPUT_DATA_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/eam_hindcast")
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
LAND_SUPPORT_DIR = OBSERVATION_ROOT / "lnd_sea_mask"           # land fraction, ELM layer thicknesses
WORKFLOW_NAME = "hind_lead_eval"

# Initialization configurations (configs/fcst_init_experiment_config.py) and seasons (4 starts each)
CONFIGS = {   # configuration key: raw case name (one case per start: INPUT_DATA_ROOT/<case name>_<YYYYMMDD>00/<ENxx>)
    "CLIM_BASE": "F20TR_ne30pg2_r05_IcoswISC30E3r5_CLIM_BASE",
    "HYB_BASE": "F20TR_ne30pg2_r05_IcoswISC30E3r5_HYB_BASE",
    "DA_ATM": "F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_ATM",
    "DA_LND": "F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_LND",
    "DA_COUPLED": "F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_COUPLED",
}
SEASONS = ("DJF", "JJA")
MAX_LEAD = 56          # lead days (8 weeks)
MAX_MEMBERS = None     # e.g. 4 for a quick test
REQUIRE_ALL_MEMBERS = True  # require every member's stream to be regenerated

VARIABLES = ("Z500", "T850", "U850", "U200", "TMQ", "PSL", "TREFHT", "PRECT", "FLUT")
# Regions: REGIONS are computed and saved (the plotted ones, PLOT_REGIONS, are set in section 7). Standard names
# (configs/s2s_config.py): GLOBAL, OCEAN, LAND, NINO3.4, MAR_CONT, CONUS, NH_POLAR. Add your own boxes to
# CUSTOM_REGIONS and name them in REGIONS: lat in -90..90, lon in 0-360 (lon0 > lon1 wraps through 0E),
# mask None, "land" or "ocean". Changing a region's box recomputes the saved regional scores.
CUSTOM_REGIONS = {
    # "EUROPE": dict(lat=(35, 72), lon=(350, 40), mask="land"),
}
REGIONS = ("GLOBAL", "LAND", "OCEAN", "NH_POLAR", "NINO3.4", "MAR_CONT", "CONUS")
CLIM_YEARS = (2001, 2014)  # common to ERA5, GPCP V3.3 (from 1998), NOAA OLR and the AMIP monthly output (FLUT to 2014)
BIASES = ("none", "amip")  # forecast anomaly vs the observed climatology ("none": drift and model bias count as error)
                           # and vs the AMIP climatology ("amip": model mean bias removed); "loo" for comparison only
AMIP_ROOT = OBSERVATION_ROOT / "E3SMv3"                 # free-running E3SMv3 DECK AMIP runs
AMIP_CASES = ("v3.LR.amip_0101", "v3.LR.amip_0151")
BASELINE = "CLIM_BASE"     # ΔACC and scorecard reference: the free-running state (no initialization information),
                           # so a difference is the skill an initialization adds beyond the prescribed SST


FORCE_COMPUTE = False


## 3. Paths and modules

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.fcst_init_experiment_config import (balanced_runs, cache_current, case_attrs, check_configs, init_dates,
                                                readiness, set_case_names)
from configs.fcst_init_experiment_config import members as config_members
from configs.output_paths import workflow_output_dirs

check_configs(CONFIGS)
set_case_names(CONFIGS)   # the readers find each configuration's cases by these names
for c, name in CONFIGS.items():
    print(f"[CASE] {c}: {name}_<YYYYMMDD>00")
data_dir, figure_dir = workflow_output_dirs(WORKFLOW_NAME, "s2s_weekly", output_root=DIAGNOSTIC_OUTPUT_ROOT)
ROOTS = dict(data_root=INPUT_DATA_ROOT, land_support_dir=LAND_SUPPORT_DIR)
print(f"[DATA] {data_dir}\n[FIGURE] {figure_dir}")

from util.fcst_init_data import region_label, regions_table, select_regions

regions = select_regions(REGIONS, CUSTOM_REGIONS)
REGION_LABELS = {r["name"]: region_label(r) for r in regions}
display(regions_table(regions))


def members_of(config, season):
    """Members of a configuration in a season (the DA runs use a different 20 DART members per season)."""
    m = config_members(config, season)
    return m if MAX_MEMBERS is None else m[:MAX_MEMBERS]


def save(fig, name):
    path = figure_dir / name
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {path}")
    if not SHOW:
        plt.close(fig)

## 4. Read input

In [4]:
status = readiness(CONFIGS, SEASONS, streams=(("atm", "daily"), ("atm", "6hourly")), data_root=INPUT_DATA_ROOT)
summary = status.groupby(["config", "season"]).agg(ready=("ready", "sum"), members=("members", "sum"))
display(summary.unstack("season"))
RUNS = balanced_runs(status, REQUIRE_ALL_MEMBERS)  # raises unless every configuration is ready in every season
print(f"[RUNS] {len(RUNS)} configuration-seasons: {RUNS}")

ready      members     
season       DJF  JJA     DJF  JJA
config                            
CLIM_BASE    160  160     160  160
DA_ATM       160  160     160  160
DA_COUPLED   160  160     160  160
DA_LND       160  160     160  160
HYB_BASE     160  160     160  160

[RUNS] 10 configuration-seasons: [('CLIM_BASE', 'DJF'), ('CLIM_BASE', 'JJA'), ('DA_ATM', 'DJF'), ('DA_ATM', 'JJA'), ('DA_COUPLED', 'DJF'), ('DA_COUPLED', 'JJA'), ('DA_LND', 'DJF'), ('DA_LND', 'JJA'), ('HYB_BASE', 'DJF'), ('HYB_BASE', 'JJA')]


## 5. Derive and save

Reference climatology per season and variable (read once, saved), then the weekly anomalies of each configuration for every treatment in `BIASES` (one read of each ensemble). Saved files are reused unless `FORCE_COMPUTE`.

In [5]:
from util.fcst_init_data import model_climatology, open_campaign, open_reference, reference_climatology
from util.fcst_init_skill import weekly_anomalies

years = list(range(CLIM_YEARS[0], CLIM_YEARS[1] + 1))
anom_path = lambda b, c, s, v: data_dir / f"anom_{b}_{c}_{s}_{v}.nc"


def cached_climatology(path, build):
    """Saved monthly-of-daily climatology of the current ``CLIM_YEARS``, else built and saved."""
    if path.exists() and not FORCE_COMPUTE:
        clim = xr.load_dataarray(path)
        source = clim.attrs.get("climatology_source", "")
        if clim.attrs.get("climatology_years") == f"{years[0]}-{years[-1]}" and (
                source.startswith("E3SMv3 AMIP") or "monthly means of the daily series" in source):
            return clim
    clim = build()
    clim.to_netcdf(path)
    print(f"[WRITE] {path}  ({clim.attrs.get('climatology_source', '')})")
    return clim


def anom_current(path, bias, config):
    """An anomaly file made with ``bias``, the current ``CLIM_YEARS`` and the current case of ``config``."""
    if not cache_current(path, config):
        return False
    with xr.open_dataset(path) as ds:
        return ds.attrs.get("bias") == bias and ds.attrs.get("clim_years") == str(CLIM_YEARS)


for season in SEASONS:
    inits = init_dates([season])[season]
    for var in VARIABLES:
        todo = {c: [b for b in BIASES if FORCE_COMPUTE or not anom_current(anom_path(b, c, season, var), b, c)]
                for c in CONFIGS if (c, season) in RUNS}
        todo = {c: bs for c, bs in todo.items() if bs}
        if not todo:
            continue
        try:
            ref = open_reference(var, inits, max_lead=MAX_LEAD, observation_root=OBSERVATION_ROOT).load()
            clim = cached_climatology(data_dir / f"clim_{season}_{var}.nc", lambda: reference_climatology(
                var, inits, years, max_lead=MAX_LEAD, observation_root=OBSERVATION_ROOT))
            mclim = cached_climatology(data_dir / f"clim_model_{season}_{var}.nc", lambda: model_climatology(
                var, inits, years, max_lead=MAX_LEAD, amip_root=AMIP_ROOT, cases=AMIP_CASES,
                land_support_dir=LAND_SUPPORT_DIR)) if any("amip" in bs for bs in todo.values()) else None
        except (FileNotFoundError, KeyError) as err:
            print(f"[SKIP] {var}: {err}")
            continue
        for c, biases in todo.items():
            model = open_campaign(c, season, var, members=members_of(c, season), max_lead=MAX_LEAD, **ROOTS).load()
            for b in biases:  # one read of the ensemble for every bias treatment
                fa, oa = weekly_anomalies(model, ref, clim, bias=b, model_climatology=mclim)
                xr.Dataset({"model": fa, "reference": oa}).assign_attrs(
                    bias=b, clim_years=str(CLIM_YEARS), reference_climatology=clim.attrs.get("climatology_source", ""),
                    model_climatology=mclim.attrs.get("climatology_source", "") if b == "amip" else "", **case_attrs(c),
                ).to_netcdf(anom_path(b, c, season, var))
                print(f"[WRITE] {anom_path(b, c, season, var)}")
            del model
anom = {(b, c, s, v): xr.load_dataset(anom_path(b, c, s, v))
        for b in BIASES for c in CONFIGS for s in SEASONS for v in VARIABLES if anom_path(b, c, s, v).exists()}
print(f"[LOAD] {len(anom)} anomaly files")

[LOAD] 180 anomaly files


## 6. Scores

For every treatment in `BIASES`: the regional lead-time skill curves (2a) and the regional-index skill and series (3a), per season and with both seasons pooled (`"ALL"`); regional pattern ACC (mean over starts) and MSSS per week; gridpoint ACC over the starts of the available seasons and its difference from `BASELINE` (on the seasons common to both).

In [ ]:
from util.fcst_init_data import region_masks
from util.fcst_init_skill import critical_acc, gridpoint_acc, msss, pattern_acc, pool_seasons

masks = None
rows = []
for (b, c, s, v), ds in anom.items():
    if masks is None:
        masks = region_masks(regions, ds.lat.values, ds.lon.values, land_support_dir=LAND_SUPPORT_DIR)
    pa = pattern_acc(ds.model, ds.reference, masks).mean("init")
    sk = msss(ds.model, ds.reference, masks)
    for r in masks.region.values:
        for w in ds.week.values:
            rows.append(dict(bias=b, config=c, season=s, variable=v, region=r, week=int(w),
                             pattern_acc=float(pa.sel(region=r, week=w)), msss=float(sk.msss.sel(region=r, week=w)),
                             rmse_anom=float(sk.rmse_anom.sel(region=r, week=w)),
                             rmse_clim=float(sk.rmse_clim.sel(region=r, week=w))))
table = pd.DataFrame(rows)
table.to_csv(data_dir / "s2s_weekly_scores.csv", index=False)
print(f"[WRITE] {data_dir / 's2s_weekly_scores.csv'}")
if len(table):
    show = table[table.region == REGIONS[0]]
    display(show.pivot_table(index=["region", "variable", "config"], columns=["bias", "season", "week"],
                             values="pattern_acc", sort=False).round(2))


def seasons_of(b, c, v):
    return [s for s in SEASONS if (b, c, s, v) in anom]


def pooled_acc(b, c, v, seasons):
    fa = pool_seasons({s: anom[b, c, s, v].model for s in seasons})
    oa = pool_seasons({s: anom[b, c, s, v].reference for s in seasons})
    return gridpoint_acc(fa, oa).assign_attrs(seasons="+".join(seasons))


_gp_cache = {}


def gp_acc(b, c, v, seasons=None):
    """Gridpoint ACC of configuration ``c`` over the starts of ``seasons`` (default: every season it has),
    computed on first use."""
    seasons = tuple(seasons or seasons_of(b, c, v))
    if (b, c, v, seasons) not in _gp_cache:
        _gp_cache[b, c, v, seasons] = pooled_acc(b, c, v, list(seasons))
    return _gp_cache[b, c, v, seasons]


n_starts = {c: sorted({sum(anom[k].sizes["init"] for k in anom if k[1] == c and k[0] == BIASES[0] and k[3] == v)
                       for v in VARIABLES} - {0}) for c in CONFIGS}
for c, n in n_starts.items():
    if n:
        print(f"[ACC] {c}: gridpoint ACC over {n} starts; 95% one-sided critical value "
              f"{', '.join(f'{critical_acc(k):.2f}' if k > 2 else 'n/a' for k in n)}")


# Regional lead-time skill (ESP-Lab 2a) and regional-index skill and series (ESP-Lab 3a), per season and pooled ("ALL")
from util.fcst_init_skill import index_skill, regional_index, window_region_summary, window_skill

lead_rows, index_rows, series_rows = [], [], []
for b in BIASES:
    for c in CONFIGS:
        for v in VARIABLES:
            have = seasons_of(b, c, v)
            sets = {s: (anom[b, c, s, v].model, anom[b, c, s, v].reference) for s in have}
            if len(have) > 1:
                sets["ALL"] = (pool_seasons({s: sets[s][0] for s in have}), pool_seasons({s: sets[s][1] for s in have}))
            for s, (fa, oa) in sets.items():
                key = dict(bias=b, config=c, season=s, variable=v)
                weeks = tuple((int(w),) for w in fa.week.values)          # each week a window of its own
                lead = window_region_summary(window_skill(fa, oa, weeks), masks)
                lead["week"] = [int(x.split()[-1]) for x in lead.window]
                lead_rows.append(lead.drop(columns="window").assign(**key))
                fi, oi = regional_index(fa, masks), regional_index(oa, masks)
                index_rows.append(index_skill(fi, oi).to_dataframe().reset_index()[
                    ["region", "week", "acc", "acc_sig", "nrmse", "rmse", "rmse_clim"]].assign(n_starts=fi.sizes["init"], **key))
                if s != "ALL":
                    series_rows.append(xr.Dataset(dict(model=fi, obs=oi)).to_dataframe().reset_index()[
                        ["region", "week", "init", "model", "obs"]].assign(**key))
lead_table = pd.concat(lead_rows, ignore_index=True)
index_table = pd.concat(index_rows, ignore_index=True)
series_table = pd.concat(series_rows, ignore_index=True)
for name, t in (("s2s_regional_lead_skill.csv", lead_table), ("s2s_index_skill.csv", index_table),
                ("s2s_index_series.csv", series_table)):
    t.to_csv(data_dir / name, index=False)
    print(f"[WRITE] {data_dir / name}")
show = index_table[(index_table.region == REGIONS[0]) & (index_table.bias == BIASES[-1])]
display(show.pivot_table(index=["variable", "config"], columns=["season", "week"], values="acc", sort=False).round(2))

## 7. Plot and show

Only the selection at the top of the next cell is plotted: the setup `PLOT_SETUP` of `PLOT_SETUPS` (bias treatments, seasons, figure types), the variables `PLOT_VARIABLES` and the regions `PLOT_REGIONS`. Change them there and rerun that cell alone (the scores above are kept in memory).

In [ ]:
# ============================ Figure settings: edit here and rerun this cell ============================
# What to plot. Every variable and region of section 2 is computed and saved; only this selection is plotted.
# A setup names the bias treatments, seasons and figure types; edit one or add your own.
# The seasons of a setup are drawn side by side in one figure (Dec and May starts compared).
#   figure types: "regional"    regional lead-time skill (ESP-Lab 2a): area-mean gridpoint ACC and regional nRMSE by
#                               lead week; rows Dec, May (and all) starts, columns ACC, nRMSE (one per variable and region)
#                 "index"       regional-index skill (ESP-Lab 3a): ACC (filled: significant) and nRMSE of the regional-mean
#                               anomaly by lead week, same layout (one per variable and region)
#                 "index_series" regional-mean anomaly by start date at the lead windows SERIES_WINDOWS (ESP-Lab 3a series)
#                 "pattern"     weekly pattern ACC and MSSS per configuration (one figure per region and metric)
#                 "acc_maps"    gridpoint ACC, weeks 1-8, per configuration (one figure per variable)
#                 "dacc_maps"   gridpoint ACC minus BASELINE in DACC_WEEKS (one figure per variable and week)
#                 "scorecard"   pattern ACC minus BASELINE, configurations x (variable, week), seasons stacked (one per region)
#                 "bias_impact" score with "amip" minus "none" (needs both in BIASES; one figure per region and metric)
PLOT_SETUPS = {
    "s2s":         dict(biases=("amip",), seasons=("DJF", "JJA"), figures=("regional", "index", "index_series", "scorecard")),
    "s2s_maps":    dict(biases=("amip",), seasons=("DJF", "JJA"), figures=("acc_maps", "dacc_maps")),
    "full_field":  dict(biases=("none",), seasons=("DJF", "JJA"), figures=("regional", "index", "index_series", "scorecard")),
    "bias_impact": dict(biases=("none", "amip"), seasons=("DJF", "JJA"), figures=("bias_impact",)),
    "all":         dict(biases=("none", "amip"), seasons=("DJF", "JJA"),
                        figures=("regional", "index", "index_series", "pattern", "acc_maps", "dacc_maps",
                                 "scorecard", "bias_impact")),
}
PLOT_SETUP = "s2s"                         # key of PLOT_SETUPS
PLOT_VARIABLES = ("TREFHT", "PRECT", "Z500")   # subset of VARIABLES
PLOT_REGIONS = ("GLOBAL", "CONUS")        # subset of REGIONS ("regional" curves, scorecard, bias impact)
INDEX_REGIONS = ("CONUS", "NINO3.4")     # subset of REGIONS ("index", "index_series"): an index needs a region whose
                                          # mean anomaly varies between starts (a global mean hardly does)
DACC_WEEKS = (2, 4)
WEEKLY_METRICS = ("pattern_acc", "msss")   # "pattern" and "bias_impact" figures
ALL_INITS = True           # "regional"/"index": add a row of all starts pooled below the Dec and May rows
SERIES_WINDOWS = ((1, 2), (3, 4))   # "index_series": lead windows (weeks), one panel each
SKILL_YLIMS = {"acc": (-1, 1)}      # "index" y ranges, e.g. {"acc": (-1, 1), "nrmse": (0, 2)}; {} = automatic
REGIONAL_YLIMS = {}                 # "regional" y ranges, keys "acc_mean", "nrmse_region"
SHOW = True          # False: save the figures without showing them
FIGURE_DPI = 200
FONT_SIZE = 18       # base font size (pt); every label of every figure scales with it
FONT_SCALES = dict(label=1.0, title=1.0, tick=0.9, legend=1.0, suptitle=1.15, cell=0.75)  # x FONT_SIZE

for name, sel in (("PLOT_REGIONS", PLOT_REGIONS), ("INDEX_REGIONS", INDEX_REGIONS)):
    if not set(sel) <= set(REGIONS):
        raise ValueError(f"{name} {sorted(set(sel) - set(REGIONS))} are not in REGIONS (section 2)")

from util.fcst_init_plotting import set_font_size
set_font_size(FONT_SIZE, **FONT_SCALES)
# =========================================================================================================

from util.fcst_init_plotting import (metric_label, plot_bias_impact, plot_config_week_maps, plot_index_series,
                                     plot_lead_skill, plot_scorecard, plot_week_maps, plot_weekly_metric)
from util.fcst_init_data import spec
from util.fcst_init_skill import relative_change
from configs.fcst_init_experiment_config import config_label


def lead_rows_of(seasons):
    return list(seasons) + (["ALL"] if ALL_INITS and len(seasons) > 1 else [])


def plot_regional(b, seasons, variables, regions):
    """ESP-Lab 2a: area mean of the gridpoint ACC, regional nRMSE."""
    t = lead_table[lead_table.bias == b]
    for v in variables:
        for r in regions:
            if len(t[(t.variable == v) & (t.region == r)]):
                save(plot_lead_skill(t, variable=v, region=r, seasons=lead_rows_of(seasons),
                                     metrics=("acc_mean", "nrmse_region"), ylims=REGIONAL_YLIMS, region_label=REGION_LABELS[r],
                                     title=f"{v} regional skill: mean gridpoint ACC and regional nRMSE (BIAS = {b})"),
                     f"s2s_regional_skill_{b}_{v}_{r}.png")


def plot_index(b, seasons, variables, regions):
    """ESP-Lab 3a: skill of the regional-mean anomaly; filled markers: ACC significant at 95%."""
    t = index_table[index_table.bias == b]
    for v in variables:
        for r in INDEX_REGIONS:
            if len(t[(t.variable == v) & (t.region == r)]):
                save(plot_lead_skill(t, variable=v, region=r, seasons=lead_rows_of(seasons), sig="acc_sig",
                                     ylims=SKILL_YLIMS, region_label=REGION_LABELS[r],
                                     title=f"{v} regional-index skill (BIAS = {b})"),
                     f"s2s_index_skill_{b}_{v}_{r}.png")


def plot_index_series_figs(b, seasons, variables, regions):
    """ESP-Lab 3a series: the regional index by start date at fixed leads."""
    t = series_table[series_table.bias == b]
    for v in variables:
        for r in INDEX_REGIONS:
            if len(t[(t.variable == v) & (t.region == r)]):
                save(plot_index_series(t, variable=v, region=r, windows=SERIES_WINDOWS, seasons=seasons,
                                       units=spec(v).units, region_label=REGION_LABELS[r],
                                       title=f"{v} regional-mean anomaly by start date (BIAS = {b})"),
                     f"s2s_index_series_{b}_{v}_{r}.png")


def plot_pattern(b, seasons, variables, regions):
    tb = table[table.bias == b]
    for r in regions:
        for m in WEEKLY_METRICS:
            if len(tb[tb.region == r]):
                save(plot_weekly_metric(tb, variables, metric=m, seasons=seasons, region=r, region_label=REGION_LABELS[r],
                                        title=f"Weekly anomaly {metric_label(m)} (BIAS = {b})"),
                     f"s2s_weekly_{b}_{r}_{m}.png")


def plot_acc_maps(b, seasons, variables, regions):
    """Gridpoint ACC pooled over the setup's seasons (more starts per gridpoint)."""
    for v in variables:
        for c in CONFIGS:
            pooled = [x for x in seasons if (b, c, x, v) in anom]
            if not pooled:
                continue
            acc = gp_acc(b, c, v, pooled)
            n = acc.attrs["n_samples"]
            r_crit = critical_acc(n) if n > 2 else np.nan
            save(plot_week_maps(acc, title=f"{config_label(c)} {v} weekly ACC, BIAS = {b} ({n} starts, {acc.attrs['seasons']}; "
                                           f"stippled > {r_crit:.2f})", stipple=acc > r_crit, label="ACC"),
                 f"s2s_acc_map_{b}_{c}_{v}.png")


def plot_dacc_maps(b, seasons, variables, regions):
    """ΔACC vs BASELINE, pooled over the seasons both have."""
    for v in variables:
        base = [x for x in seasons if (b, BASELINE, x, v) in anom]
        diff, pooled = {}, set()
        for c in CONFIGS:
            common = [x for x in seasons if (b, c, x, v) in anom and x in base]
            if c == BASELINE or not common:
                continue
            # same starts for both, so a configuration missing a season is not compared with a larger sample
            diff[c] = gp_acc(b, c, v, common) - gp_acc(b, BASELINE, v, common)
            pooled.add("+".join(common))
        for w in DACC_WEEKS if diff else ():
            save(plot_config_week_maps(diff, week=w, title=f"{v} ΔACC vs {config_label(BASELINE)}, week {w}, BIAS = {b} "
                                                           f"(common starts: {', '.join(sorted(pooled))})",
                                       levels=np.linspace(-0.6, 0.6, 13), label="ΔACC"), f"s2s_dacc_{b}_{v}_week{w}.png")


def plot_scorecards(b, seasons, variables, regions):
    """Seasons stacked in one figure on one colour scale."""
    for r in regions:
        changes = {}
        for s in seasons:
            t = table[(table.bias == b) & (table.season == s) & (table.region == r) & table.variable.isin(variables)]
            if BASELINE not in set(t.config):
                continue
            wide = t.pivot_table(index="config", columns=["variable", "week"], values="pattern_acc", sort=False)
            wide = wide.reindex(index=[c for c in CONFIGS if c in wide.index],
                                columns=[v for v in variables if v in wide.columns.get_level_values(0)], level="variable")
            changes[s] = relative_change(wide, BASELINE, "pattern_acc").drop(index=BASELINE)
        if changes:
            save(plot_scorecard(changes, higher_is_better=True, units="ΔACC",
                                title=f"Pattern ACC minus {config_label(BASELINE)}, {REGION_LABELS[r]} (BIAS = {b})"),
                 f"s2s_scorecard_{b}_{r}.png")


def plot_bias_impacts(b, seasons, variables, regions):
    """``amip`` minus ``none``: drawn once (with the last bias treatment of the setup)."""
    if not {"none", "amip"} <= set(BIASES) or b != setup["biases"][-1]:
        return
    for r in regions:
        for m in WEEKLY_METRICS:
            if len(table[table.region == r]):
                save(plot_bias_impact(table, variables, metric=m, seasons=seasons, region=r,
                                      region_label=REGION_LABELS[r]), f"s2s_bias_impact_{r}_{m}.png")


FIGURES = {"regional": plot_regional, "index": plot_index, "index_series": plot_index_series_figs, "pattern": plot_pattern,
           "acc_maps": plot_acc_maps, "dacc_maps": plot_dacc_maps, "scorecard": plot_scorecards,
           "bias_impact": plot_bias_impacts}
setup = PLOT_SETUPS[PLOT_SETUP]
unknown = [(k, x) for k, xs, ok in (("figure", setup["figures"], FIGURES), ("variable", PLOT_VARIABLES, VARIABLES),
                                    ("region", PLOT_REGIONS, REGIONS), ("bias", setup["biases"], BIASES),
                                    ("season", setup["seasons"], SEASONS)) for x in xs if x not in ok]
if unknown:
    raise ValueError(f"not available (not computed or unknown): {unknown}")
print(f"[PLOT] setup {PLOT_SETUP!r}: {setup}\n[PLOT] variables {PLOT_VARIABLES}; regions {PLOT_REGIONS}")
for b in setup["biases"]:
    for kind in setup["figures"]:
        FIGURES[kind](b, list(setup["seasons"]), list(PLOT_VARIABLES), list(PLOT_REGIONS))